In [18]:
import numpy as np

# Set seed
np.random.seed(42)

# RNN dimensions
n_x = 3      # Input size
n_h = 5      # Hidden state size
n_y = 2      # Output size
T   = 4      # Sequence length

# Initialize weights
W_xh = np.random.randn(n_h, n_x) * 0.01
W_hh = np.random.randn(n_h, n_h) * 0.01
b_h  = np.zeros((n_h, 1))

W_hy = np.random.randn(n_y, n_h) * 0.01
b_y  = np.zeros((n_y, 1))

# Utility functions
def softmax(z):
    e_z = np.exp(z - np.max(z))
    return e_z / np.sum(e_z, axis=0, keepdims=True)

def compute_loss(y_hat_seq, y_seq):
    loss = 0
    for t in range(len(y_seq)):
        y_true = np.zeros_like(y_hat_seq[t])
        y_true[y_seq[t], 0] = 1
        loss -= np.sum(y_true * np.log(y_hat_seq[t] + 1e-9))  # cross-entropy
    return loss

# Forward pass
def forward_pass(x_seq, h_prev):
    h_cache, a_cache, y_hat_seq, z_cache = {}, {}, {}, {}
    for t in range(len(x_seq)):
        x = x_seq[t]
        a = W_xh @ x + W_hh @ h_prev + b_h
        h = np.tanh(a)
        z = W_hy @ h + b_y
        y_hat = softmax(z)
        
        h_cache[t] = h
        a_cache[t] = a
        z_cache[t] = z
        y_hat_seq[t] = y_hat

        h_prev = h
    return y_hat_seq, h_cache, a_cache, z_cache

# BPTT
def bptt(x_seq, y_seq, y_hat_seq, h_cache, a_cache):
    global W_xh, W_hh, b_h, W_hy, b_y
    dW_xh = np.zeros_like(W_xh)
    dW_hh = np.zeros_like(W_hh)
    db_h  = np.zeros_like(b_h)
    dW_hy = np.zeros_like(W_hy)
    db_y  = np.zeros_like(b_y)
    dh_next = np.zeros((n_h, 1))

    for t in reversed(range(len(x_seq))):
        dy = y_hat_seq[t].copy()
        dy[y_seq[t]] -= 1

        dW_hy += dy @ h_cache[t].T
        db_y  += dy

        dh = W_hy.T @ dy + dh_next
        da = dh * (1 - np.tanh(a_cache[t]) ** 2)

        dW_xh += da @ x_seq[t].T
        dW_hh += da @ h_cache[t-1].T if t != 0 else da @ np.zeros_like(h_cache[t]).T
        db_h  += da

        dh_next = W_hh.T @ da

    return dW_xh, dW_hh, db_h, dW_hy, db_y

# Parameter update
def update_params(dW_xh, dW_hh, db_h, dW_hy, db_y, learning_rate=0.01):
    global W_xh, W_hh, b_h, W_hy, b_y
    W_xh -= learning_rate * dW_xh
    W_hh -= learning_rate * dW_hh
    b_h  -= learning_rate * db_h
    W_hy -= learning_rate * dW_hy
    b_y  -= learning_rate * db_y


In [19]:
# Generate dummy training data
num_epochs = 100
learning_rate = 0.01

for epoch in range(num_epochs):
    # Simulated input and target sequences
    x_seq = [np.random.randn(n_x, 1) for _ in range(T)]
    y_seq = [np.random.randint(0, n_y) for _ in range(T)]

    # Initial hidden state
    h0 = np.zeros((n_h, 1))

    # Forward pass
    y_hat_seq, h_cache, a_cache, z_cache = forward_pass(x_seq, h0)

    # Compute loss
    loss = compute_loss(y_hat_seq, y_seq)

    # Backward pass (BPTT)
    dW_xh, dW_hh, db_h, dW_hy, db_y = bptt(x_seq, y_seq, y_hat_seq, h_cache, a_cache)

    # Gradient descent update
    update_params(dW_xh, dW_hh, db_h, dW_hy, db_y, learning_rate)

    # Logging
    if epoch % 10 == 0 or epoch == num_epochs - 1:
        print(f"Epoch {epoch+1}/{num_epochs} — Loss: {loss:.4f}")

Epoch 1/100 — Loss: 2.7726
Epoch 11/100 — Loss: 2.7768
Epoch 21/100 — Loss: 2.7768
Epoch 31/100 — Loss: 2.7751
Epoch 41/100 — Loss: 2.7566
Epoch 51/100 — Loss: 2.7418
Epoch 61/100 — Loss: 2.7102
Epoch 71/100 — Loss: 2.7761
Epoch 81/100 — Loss: 2.6316
Epoch 91/100 — Loss: 2.8100
Epoch 100/100 — Loss: 2.7358
